# 17 概率校准实验

每个外层测试组完全隔离。校准器只用外层训练集内部的分组 OOF 概率拟合；主方法为 Platt/sigmoid，isotonic 仅作补充比较。


In [ ]:
from pathlib import Path
from datetime import datetime
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
BASE_DIR = Path(r"D:\tabpfn-demo")
RANDOM_STATE = 42
TARGET_ORDER = ["EMR", "QDD", "HCL", "HRA"]

def normal(s):
    return re.sub(r"[^a-z0-9\u4e00-\u9fff]", "", str(s).lower())

def find_col(cols, options, required=True):
    lookup = {normal(c): c for c in cols}
    for x in options:
        if normal(x) in lookup: return lookup[normal(x)]
    for c in cols:
        if any(normal(x) in normal(c) for x in options): return c
    if required: raise KeyError(f"未找到列 {options}；实际列：{list(cols)}")
    return None

candidates = [Path(r"E:\桌面\total_dataset_final.xlsx"), Path(r"E:\桌面\total_dataset.xlsx"),
              BASE_DIR / "total_dataset_final.xlsx", BASE_DIR / "total_dataset.xlsx"]
candidates += sorted(BASE_DIR.glob("*.xlsx"))
EXCEL_PATH = next((p for p in candidates if p.exists() and not p.name.startswith("~$")), None)
if EXCEL_PATH is None: raise FileNotFoundError("请设置 EXCEL_PATH 为数据集路径。")
raw = pd.read_excel(EXCEL_PATH, sheet_name=0)
TARGET_COL = find_col(raw.columns, ["驱动形式", "驱动类型", "drive_type", "label", "类别"])
FEATURE_COLS = [
    find_col(raw.columns, ["输出角度", "angle"]),
    find_col(raw.columns, ["额定转速", "rated speed"]),
    find_col(raw.columns, ["峰值转速", "peak speed"]),
    find_col(raw.columns, ["额定力矩", "rated torque"]),
    find_col(raw.columns, ["峰值力矩", "peak torque"]),
    find_col(raw.columns, ["额定功率", "rated power"]),
    find_col(raw.columns, ["峰值功率", "peak power"]),
]
MANUFACTURER_COL = find_col(raw.columns, ["厂商", "制造商", "manufacturer", "品牌"], False)
MODEL_COL = find_col(raw.columns, ["型号", "model"], False)
FAMILY_COL = find_col(raw.columns, ["产品系列", "系列", "family"], False)
df = raw.copy()
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.upper()
df = df[df[TARGET_COL].isin(TARGET_ORDER)].reset_index(drop=True)
for col in FEATURE_COLS: df[col] = pd.to_numeric(df[col], errors="coerce")
X = df[FEATURE_COLS]
encoder = LabelEncoder().fit(TARGET_ORDER)
y = encoder.transform(df[TARGET_COL])
CLASS_NAMES = list(encoder.classes_)
print(f"数据：{EXCEL_PATH}; 样本数={len(df)}; 类别={df[TARGET_COL].value_counts().to_dict()}")

def make_model():
    try:
        from xgboost import XGBClassifier
        model = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_estimators=350,
            max_depth=3, learning_rate=0.05, subsample=.9, colsample_bytree=.9,
            random_state=RANDOM_STATE, n_jobs=-1)
        backend = "XGBoost"
    except ImportError:
        model = RandomForestClassifier(n_estimators=600, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1)
        backend = "RandomForest fallback; 正式实验请安装 xgboost 后重跑"
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)]), backend

def full_proba(model, part):
    p = np.asarray(model.predict_proba(part))
    classes = np.asarray(
        getattr(
            model,
            "_ec_global_classes_",
            getattr(model.named_steps["model"], "_ec_global_classes_", model.named_steps["model"].classes_),
        ),
        dtype=int,
    )
    out = np.zeros((len(part), len(CLASS_NAMES)))
    cols = min(p.shape[1], len(classes))
    out[:, classes[:cols]] = p[:, :cols]
    return out / np.maximum(out.sum(axis=1, keepdims=True), 1e-12)

def topk(y_true, p, k):
    return np.mean([t in np.argsort(row)[-k:] for t, row in zip(y_true, p)])

def fit_model(model, Xtr, ytr):
    ytr = np.asarray(ytr, dtype=int)
    classes = np.unique(ytr)
    y_local = np.zeros(len(ytr), dtype=int) if len(classes) == 1 else np.searchsorted(classes, ytr)
    model.fit(Xtr, y_local)
    setattr(model, "_ec_global_classes_", classes)
    if hasattr(model, "named_steps") and "model" in model.named_steps:
        setattr(model.named_steps["model"], "_ec_global_classes_", classes)
    return model

def predict_global(model, part):
    pred = np.asarray(model.predict(part), dtype=int)
    classes = np.asarray(
        getattr(
            model,
            "_ec_global_classes_",
            getattr(model.named_steps["model"], "_ec_global_classes_", model.named_steps["model"].classes_),
        ),
        dtype=int,
    )
    pred = np.clip(pred, 0, len(classes) - 1)
    return classes[pred]

OUTPUT_DIR = BASE_DIR / f"17_probability_calibration_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def build_group_ids(frame):
    n = len(frame); parent = list(range(n))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def join(a, b):
        a, b = root(a), root(b)
        if a != b: parent[b] = a
    maker = frame[MANUFACTURER_COL].fillna("unknown").astype(str).str.strip() if MANUFACTURER_COL else pd.Series("unknown", index=frame.index)
    if FAMILY_COL:
        family = frame[FAMILY_COL].fillna("").astype(str).str.strip()
    elif MODEL_COL:
        family = frame[MODEL_COL].fillna("").astype(str).str.upper().str.replace(r"[0-9].*$", "", regex=True)
    else:
        family = pd.Series("", index=frame.index)
    # 同系列和七维性能完全相同的样本通过并查集合并，避免信息泄漏。
    family_key = (maker + "|" + family).where(family.ne(""), "")
    duplicate_key = frame[FEATURE_COLS].round(8).astype(str).agg("|".join, axis=1)
    for key_series in (family_key, duplicate_key):
        seen = {}
        for i, key in enumerate(key_series):
            if key in seen and key: join(i, seen[key])
            elif key: seen[key] = i
    roots = [root(i) for i in range(n)]
    remap = {r: f"G{i:03d}" for i, r in enumerate(sorted(set(roots)))}
    group = np.array([remap[r] for r in roots])
    audit = pd.DataFrame({"manufacturer_id": maker, "family_id": family,
        "duplicate_cluster_id": duplicate_key, "group_id": group, "drive_type": frame[TARGET_COL]})
    return group, audit

groups, group_audit = build_group_ids(df)
group_counts = pd.DataFrame({"y": y, "g": groups}).groupby("y").g.nunique()
N_SPLITS = min(5, int(group_counts.min()))
if N_SPLITS < 3: raise ValueError(f"最小类别仅 {N_SPLITS} 个独立组；请人工补充 family_id 后再运行。")
print("每类独立组数：", group_counts.to_dict(), "；外层分组折数：", N_SPLITS)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss

def brier(y_true,p): return np.mean(np.sum((p-np.eye(p.shape[1])[y_true])**2,axis=1))
def ece(y_true,p,bins=10):
    conf=p.max(1); pred=p.argmax(1); ans=0.
    for lo,hi in zip(np.linspace(0,1,bins+1)[:-1],np.linspace(0,1,bins+1)[1:]):
        mask=(conf>=lo)&(conf <=hi if hi==1 else conf<hi)
        if mask.any(): ans += mask.mean()*abs((pred[mask]==y_true[mask]).mean()-conf[mask].mean())
    return ans
def inner_oof(Xtr,ytr,gtr):
    group_counts = pd.DataFrame({"y":ytr,"g":gtr}).groupby("y").g.nunique()
    min_class_groups = int(group_counts.min())
    n_groups = int(pd.Series(gtr).nunique())
    if n_groups < 2: raise ValueError("训练折独立组数小于 2，不能进行分组校准。")
    if min_class_groups >= 2:
        n = min(3, min_class_groups)
        cv = StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=RANDOM_STATE)
        split_iter = cv.split(Xtr,ytr,gtr)
        status = f"stratified_group_{n}fold"
    else:
        n = min(3, n_groups)
        cv = GroupKFold(n_splits=n)
        split_iter = cv.split(Xtr,ytr,gtr)
        status = f"group_{n}fold_fallback_min_class_groups_{min_class_groups}"
    out=np.zeros((len(Xtr),4))
    for a,b in split_iter:
        m,_=make_model(); fit_model(m,Xtr.iloc[a],ytr[a]); out[b]=full_proba(m,Xtr.iloc[b])
    return out,status
def calibrate(oof,ytr,method):
    if method=="raw": return None
    if method=="platt": return LogisticRegression(max_iter=3000,multi_class="multinomial",random_state=RANDOM_STATE).fit(np.log(np.clip(oof,1e-8,1)),ytr)
    return [IsotonicRegression(out_of_bounds="clip").fit(oof[:,i],(ytr==i).astype(int)) for i in range(4)]
def transform(cal,p,method):
    if method=="raw": return p
    if method=="platt":
        q_local = cal.predict_proba(np.log(np.clip(p,1e-8,1)))
        q = np.zeros((len(p),4))
        q[:,np.asarray(cal.classes_,dtype=int)] = q_local
        return q/np.maximum(q.sum(1,keepdims=True),1e-12)
    q=np.column_stack([c.predict(p[:,i]) for i,c in enumerate(cal)])
    return q/np.maximum(q.sum(1,keepdims=True),1e-12)


In [ ]:
rows=[]; curve=[]
outer=StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE)
for fold,(tr,te) in enumerate(outer.split(X,y,groups),1):
    Xtr=X.iloc[tr].reset_index(drop=True); oof,calibration_cv=inner_oof(Xtr,y[tr],groups[tr])
    m,backend=make_model(); fit_model(m,Xtr,y[tr]); rawp=full_proba(m,X.iloc[te])
    for method in ["raw","platt","isotonic"]:
        p=transform(calibrate(oof,y[tr],method),rawp,method)
        rows.append({"fold":fold,"method":method,"backend":backend,"calibration_cv":calibration_cv,"brier":brier(y[te],p),"ece":ece(y[te],p),
          "nll":log_loss(y[te],p,labels=np.arange(4)),"accuracy":accuracy_score(y[te],p.argmax(1)),
          "top2_accuracy":topk(y[te],p,2),"top3_accuracy":topk(y[te],p,3)})
        for i,c in enumerate(CLASS_NAMES):
            curve.extend({"method":method,"class":c,"p":q,"truth":int(t==i)} for q,t in zip(p[:,i],y[te]))
metrics=pd.DataFrame(rows); curves=pd.DataFrame(curve)
NUMERIC_METRICS=["brier","ece","nll","accuracy","top2_accuracy","top3_accuracy"]
summary=metrics.groupby("method")[NUMERIC_METRICS].agg(["mean","std"])
protocol_counts=metrics.groupby(["method","calibration_cv"]).size().rename("fold_count").reset_index()
metrics.to_csv(OUTPUT_DIR/"outer_fold_calibration_metrics.csv",index=False,encoding="utf-8-sig"); summary.to_csv(OUTPUT_DIR/"calibration_summary.csv",encoding="utf-8-sig")
protocol_counts.to_csv(OUTPUT_DIR/"calibration_cv_protocol_counts.csv",index=False,encoding="utf-8-sig")
display(summary); display(protocol_counts)
fig,axs=plt.subplots(1,4,figsize=(16,4),sharex=True,sharey=True)
for ax,c in zip(axs,CLASS_NAMES):
    ax.plot([0,1],[0,1],"--",color="gray")
    for method in metrics.method.unique():
        s=curves[(curves["class"]==c)&(curves.method==method)]; bins=pd.cut(s.p,np.linspace(0,1,11),include_lowest=True)
        pts=s.groupby(bins,observed=True).agg(p=("p","mean"),truth=("truth","mean"))
        ax.plot(pts.p,pts.truth,"o-",label=method)
    ax.set_title(c); ax.set_xlabel("预测概率"); ax.grid(alpha=.25)
axs[0].set_ylabel("经验发生率"); axs[-1].legend(); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"classwise_reliability.png",dpi=220); plt.show()
print("若 Platt 同时降低 Brier/ECE 且 Top-2/Top-3 无实质下降，后续 Top-k 使用校准概率；否则保留原始概率。",OUTPUT_DIR)
